### PHẦN 1

In [35]:
import sqlite3
import pandas as pd

# 1. Khởi tạo database trong bộ nhớ (memory)
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# 2. Tạo bảng và chèn dữ liệu student [cite: 2, 3]
cursor.execute('''
CREATE TABLE student (
    student_id INTEGER,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
)
''')

students_data = [
    (1, 'Nguyen Minh Hoang', 'May Tinh', 12, 6.7),
    (2, 'Tran Thi Lan', 'Kinh Te', 34, 9.2),
    (3, 'Pham Van Nam', 'Toan Tin', None, 7.9),
    (4, 'Le Thanh Huyen', 'Toan Tin', 20, 7.2),
    (5, 'Vu Quoc Anh', 'May Tinh', 24, 8.0),
    (6, 'Dang Thuy Linh', 'May Tinh', 24, 5.5),
    (7, 'Bui Tien Dung', 'Kinh Te', 34, 9.2),
    (8, 'Ho Ngoc Mai', 'Toan Tin', 20, 8.8),
    (9, 'Duong Huu Phuc', 'Kinh Te', None, 7.2),
    (10, 'Cao Thi Hanh', 'May Tinh', None, 7.0)
]
cursor.executemany('INSERT INTO student VALUES (?,?,?,?,?)', students_data)

# 3. Tạo bảng và chèn dữ liệu course [cite: 2, 4]
cursor.execute('''
CREATE TABLE course (
    id INTEGER,
    course_name TEXT
)
''')

courses_data = [
    (12, 'Giai tich'),
    (34, 'Thong ke'),
    (26, 'Tin hoc')
]
cursor.executemany('INSERT INTO course VALUES (?,?)', courses_data)

def query(sql):
    return pd.read_sql_query(sql, conn)

In [45]:
# Tích Descartes
sql_descartes = "SELECT * FROM student CROSS JOIN course"
print(" Tích Descartes ")
display(query(sql_descartes))

 Tích Descartes 


,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,12,Giai tich
1,1,Nguyen Minh Hoang,May Tinh,12,6.7,34,Thong ke
2,1,Nguyen Minh Hoang,May Tinh,12,6.7,26,Tin hoc
3,2,Tran Thi Lan,Kinh Te,34,9.2,12,Giai tich
4,2,Tran Thi Lan,Kinh Te,34,9.2,34,Thong ke
5,2,Tran Thi Lan,Kinh Te,34,9.2,26,Tin hoc
6,3,Pham Van Nam,Toan Tin,12,7.9,12,Giai tich
7,3,Pham Van Nam,Toan Tin,12,7.9,34,Thong ke
8,3,Pham Van Nam,Toan Tin,12,7.9,26,Tin hoc
9,7,Bui Tien Dung,Kinh Te,34,9.2,12,Giai tich


In [37]:
# INNER JOIN
sql_inner = """
SELECT s.*, c.course_name
FROM student s
INNER JOIN course c ON s.course_id = c.id
"""
display(query(sql_inner))

,student_id,name,class,course_id,score,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,Giai tich
1,2,Tran Thi Lan,Kinh Te,34,9.2,Thong ke
2,7,Bui Tien Dung,Kinh Te,34,9.2,Thong ke


### PHẦN 2

In [46]:
# 1. Cập nhật các giá trị course_id còn thiếu (NULL) thành 12
cursor.execute('''
UPDATE student
SET course_id = 12
WHERE course_id IS NULL
''')

# 2. Loại bỏ các bản ghi tham gia môn học không tồn tại trong bảng course
cursor.execute('''
DELETE FROM student
WHERE course_id NOT IN (SELECT id FROM course)
''')

conn.commit()
print(" Dữ liệu sau khi làm sạch ")
display(query("SELECT * FROM student"))

 Dữ liệu sau khi làm sạch 


,student_id,name,class,course_id,score
0,1,Nguyen Minh Hoang,May Tinh,12,6.7
1,2,Tran Thi Lan,Kinh Te,34,9.2
2,3,Pham Van Nam,Toan Tin,12,7.9
3,7,Bui Tien Dung,Kinh Te,34,9.2
4,9,Duong Huu Phuc,Kinh Te,12,7.2
5,10,Cao Thi Hanh,May Tinh,12,7.0


In [48]:
# a. Tổng số sinh viên và điểm trung bình từng lớp
sql_2a = """
SELECT class,
       COUNT(student_id) AS total_students,
       ROUND(AVG(score), 2) AS average_score
FROM student
GROUP BY class
"""
print(" Thống kê theo Lớp ")
display(query(sql_2a))

 Thống kê theo Lớp 


,class,total_students,average_score
0,Kinh Te,3,8.53
1,May Tinh,2,6.85
2,Toan Tin,1,7.90


In [49]:
# b. Tổng số sinh viên và điểm trung bình từng môn học
sql_2b = """
SELECT c.course_name,
       COUNT(s.student_id) AS total_students,
       ROUND(AVG(s.score), 2) AS average_score
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
"""
print(" Thống kê theo Môn học ")
display(query(sql_2b))

 Thống kê theo Môn học 


,course_name,total_students,average_score
0,Giai tich,4,7.2
1,Thong ke,2,9.2


In [50]:
# c. Phân loại thi đua
sql_2c = """
SELECT c.course_name,
       ROUND(AVG(s.score), 2) AS avg_score,
       CASE
            WHEN AVG(s.score) >= 9.0 THEN 'Xuất sắc'
            WHEN AVG(s.score) >= 6.0 THEN 'Tốt'
            ELSE 'Kém'
       END AS classification
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
"""
print(" Phân loại thi đua theo môn học ")
display(query(sql_2c))

 Phân loại thi đua theo môn học 


,course_name,avg_score,classification
0,Giai tich,7.2,Tốt
1,Thong ke,9.2,Xuất sắc


### PHẦN 3


In [51]:
# a. Xếp hạng theo điểm số toàn trường
# Sử dụng RANK() để xếp hạng toàn bộ sinh viên theo điểm giảm dần
sql_3a = """
WITH RankedStudents AS (
    SELECT *,
           RANK() OVER (ORDER BY score DESC) as rank_high,
           RANK() OVER (ORDER BY score ASC) as rank_low
    FROM student
)
SELECT * FROM RankedStudents
WHERE rank_high <= 3 OR rank_low <= 3
ORDER BY score DESC
"""
print(" Top 3 Cao nhất & Thấp nhất (Toàn trường) ")
display(query(sql_3a))

 Top 3 Cao nhất & Thấp nhất (Toàn trường) 


,student_id,name,class,course_id,score,rank_high,rank_low
0,2,Tran Thi Lan,Kinh Te,34,9.2,1,5
1,7,Bui Tien Dung,Kinh Te,34,9.2,1,5
2,3,Pham Van Nam,Toan Tin,12,7.9,3,4
3,9,Duong Huu Phuc,Kinh Te,12,7.2,4,3
4,10,Cao Thi Hanh,May Tinh,12,7.0,5,2
5,1,Nguyen Minh Hoang,May Tinh,12,6.7,6,1


In [52]:
# b. Xếp hạng theo điểm số theo lớp học
sql_3b = """
WITH ClassRank AS (
    SELECT *,
           RANK() OVER (PARTITION BY class ORDER BY score DESC) as rank_high,
           RANK() OVER (PARTITION BY class ORDER BY score ASC) as rank_low
    FROM student
)
SELECT * FROM ClassRank
WHERE rank_high <= 3 OR rank_low <= 3
ORDER BY class, rank_high
"""
print(" Top 3 Cao nhất & Thấp nhất (Theo Lớp) ")
display(query(sql_3b))

 Top 3 Cao nhất & Thấp nhất (Theo Lớp) 


,student_id,name,class,course_id,score,rank_high,rank_low
0,2,Tran Thi Lan,Kinh Te,34,9.2,1,2
1,7,Bui Tien Dung,Kinh Te,34,9.2,1,2
2,9,Duong Huu Phuc,Kinh Te,12,7.2,3,1
3,10,Cao Thi Hanh,May Tinh,12,7.0,1,2
4,1,Nguyen Minh Hoang,May Tinh,12,6.7,2,1
5,3,Pham Van Nam,Toan Tin,12,7.9,1,1


In [53]:
# c. Xếp hạng theo điểm số mã môn học
sql_3c = """
WITH CourseRank AS (
    SELECT s.*, c.course_name,
           RANK() OVER (PARTITION BY s.course_id ORDER BY s.score DESC) as rank_high,
           RANK() OVER (PARTITION BY s.course_id ORDER BY s.score ASC) as rank_low
    FROM student s
    JOIN course c ON s.course_id = c.id
)
SELECT * FROM CourseRank
WHERE rank_high <= 3 OR rank_low <= 3
ORDER BY course_name, rank_high
"""
print(" Top 3 Cao nhất & Thấp nhất (Theo Môn học) ")
display(query(sql_3c))

 Top 3 Cao nhất & Thấp nhất (Theo Môn học) 


,student_id,name,class,course_id,score,course_name,rank_high,rank_low
0,3,Pham Van Nam,Toan Tin,12,7.9,Giai tich,1,4
1,9,Duong Huu Phuc,Kinh Te,12,7.2,Giai tich,2,3
2,10,Cao Thi Hanh,May Tinh,12,7.0,Giai tich,3,2
3,1,Nguyen Minh Hoang,May Tinh,12,6.7,Giai tich,4,1
4,2,Tran Thi Lan,Kinh Te,34,9.2,Thong ke,1,1
5,7,Bui Tien Dung,Kinh Te,34,9.2,Thong ke,1,1


### PHẦN 4

In [55]:
# 1. Thêm cột mới với kiểu dữ liệu DATETIME
try:
    cursor.execute("ALTER TABLE student ADD COLUMN graduation_date DATETIME")
    conn.commit()
    print(" Đã thêm cột graduation_date thành công ")
except sqlite3.OperationalError:
    print("Cột đã tồn tại.")

Cột đã tồn tại.


In [56]:
# 2. Cập nhật ngày tốt nghiệp: Ngày hiện tại + (Xếp hạng theo điểm số) ngày
sql_update_date = """
WITH Ranked AS (
    SELECT student_id,
           RANK() OVER (ORDER BY score DESC) as rnk
    FROM student
)
UPDATE student
SET graduation_date = (
    SELECT date('now', '+' || rnk || ' days')
    FROM Ranked
    WHERE Ranked.student_id = student.student_id
)
"""

cursor.execute(sql_update_date)
conn.commit()

# Hiển thị kết quả cuối cùng
print("--- Kết quả bảng Student sau khi cập nhật ngày tốt nghiệp ---")
display(query("SELECT student_id, name, score, graduation_date FROM student ORDER BY score DESC"))

--- Kết quả bảng Student sau khi cập nhật ngày tốt nghiệp ---


,student_id,name,score,graduation_date
0,2,Tran Thi Lan,9.2,2026-04-01
1,7,Bui Tien Dung,9.2,2026-04-01
2,3,Pham Van Nam,7.9,2026-04-03
3,9,Duong Huu Phuc,7.2,2026-04-04
4,10,Cao Thi Hanh,7.0,2026-04-05
5,1,Nguyen Minh Hoang,6.7,2026-04-06
